<a href="https://colab.research.google.com/github/Younessamadiaz/fragment-flow-nept-darp-roanne/blob/main/DARP_DATA-Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install osmnx mimesis geopy openpyxl geopandas networkx shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 32.6 MB/s eta 0:00:00


In [2]:
# ============================================================
# ROANNE HEALTHCARE DARP - BATCH INSTANCE GENERATOR
# Generates one database per instance and one ZIP containing all results.
# Paste/run this after your helper functions are defined, or append it at the
# end of your current notebook/script.
# ============================================================

import os
import math
import random
import shutil
import gc
import gzip
import subprocess
import sys
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

try:
    import openpyxl  # required by pandas ExcelWriter
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ------------------------------------------------------------
# Standalone helper functions copied/adapted from the original code
# ------------------------------------------------------------
CATCHMENT_RADIUS_BY_ACTIVITY_KM = {
    "dialysis": 8,
    "oncology": 10,
    "other": 12,
    "external": 18,
}
STRICT_PATIENT_CATCHMENT = True
NEAREST_HOSPITAL_TOLERANCE_KM = 1.50
MAX_LOCATION_ATTEMPTS = 2000


def minute_to_datetime(service_date, minute_of_day):
    base = datetime.strptime(service_date, "%Y-%m-%d")
    return base + timedelta(minutes=int(minute_of_day))


def datetime_to_minute(dt, service_date):
    base = datetime.strptime(service_date, "%Y-%m-%d")
    return int((dt - base).total_seconds() // 60)


def hhmm_to_min(hhmm):
    h, m = map(int, hhmm.split(":"))
    return h * 60 + m


def min_to_hhmm(m):
    m = int(m) % (24 * 60)
    return f"{m // 60:02d}:{m % 60:02d}"


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1 = math.radians(float(lat1))
    lon1 = math.radians(float(lon1))
    lat2 = math.radians(float(lat2))
    lon2 = math.radians(float(lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    return r * c


def road_distance_est_km(lat1, lon1, lat2, lon2):
    return haversine_km(lat1, lon1, lat2, lon2) * ROAD_FACTOR


def split_capacity_by_weights(total, weights):
    total = int(total)
    weights = np.array(weights, dtype=float)
    if total < 0:
        raise ValueError("total must be non-negative.")
    if np.isclose(weights.sum(), 0):
        raise ValueError("weights must have positive sum.")
    raw = total * weights / weights.sum()
    base = np.floor(raw).astype(int)
    remainder = total - int(base.sum())
    if remainder > 0:
        order = np.argsort(-(raw - base))
        for idx in order[:remainder]:
            base[idx] += 1
    return base.tolist()


def add_activity_schedule(
    rows,
    hospital_id,
    activity_type,
    daily_capacity,
    slot_times,
    source_quality,
    source_note,
    slot_weights=None,
    slot_capacities=None,
):
    if slot_capacities is not None:
        slot_caps = [int(x) for x in slot_capacities]
        if sum(slot_caps) != int(daily_capacity):
            raise ValueError(
                f"{hospital_id}-{activity_type}: daily_capacity={daily_capacity}, "
                f"sum(slot_capacities)={sum(slot_caps)}."
            )
    elif slot_weights is not None:
        slot_caps = split_capacity_by_weights(daily_capacity, slot_weights)
    else:
        raise ValueError("Provide slot_weights or slot_capacities.")

    if len(slot_times) != len(slot_caps):
        raise ValueError("slot_times and slot capacities must have same length.")

    for slot_time, slot_capacity in zip(slot_times, slot_caps):
        rows.append({
            "hospital_id": hospital_id,
            "activity_type": activity_type,
            "daily_capacity": int(daily_capacity),
            "slot_time": slot_time,
            "slot_start_min": hhmm_to_min(slot_time),
            "slot_capacity": int(slot_capacity),
            "source_quality": source_quality,
            "source_note": source_note,
        })


def scaled_capacity(value, share):
    return int(math.ceil(float(value) * float(share)))


def compute_max_ride_time_min(direct_time_min, priority_class):
    if priority_class == "high":
        return int(max(25, math.ceil(direct_time_min * 1.35)))
    if priority_class == "medium":
        return int(max(35, math.ceil(direct_time_min * 1.55)))
    return int(max(45, math.ceil(direct_time_min * 1.80)))


def compatible(requirement, vehicle_type):
    if requirement == "prescription_medicale":
        return int(vehicle_type in ["taxi_conventionne", "vsl"])
    if requirement == "vsl_required":
        return int(vehicle_type == "vsl")
    if requirement == "premium":
        return int(vehicle_type in ["vtc_premium", "taxi_conventionne"])
    if requirement == "standard_assis":
        return int(vehicle_type in ["taxi_conventionne", "vsl", "vtc_premium"])
    return 0


def random_point_around(lat, lon, radius_km):
    r = float(radius_km) * math.sqrt(random.random())
    theta = random.uniform(0, 2 * math.pi)
    delta_lat = (r * math.cos(theta)) / 111.0
    delta_lon = (r * math.sin(theta)) / (111.0 * math.cos(math.radians(lat)))
    return float(lat + delta_lat), float(lon + delta_lon)


def distance_to_hospital_km(lat, lon, hospital_row):
    return road_distance_est_km(
        lat,
        lon,
        float(hospital_row["lat"]),
        float(hospital_row["lon"]),
    )


def closest_hospital_for_point(lat, lon, hospitals_df):
    rows = []
    for _, h in hospitals_df.iterrows():
        rows.append({
            "hospital_id": h["hospital_id"],
            "hospital_name": h["hospital_name"],
            "distance_km": distance_to_hospital_km(lat, lon, h),
        })
    df = pd.DataFrame(rows).sort_values("distance_km").reset_index(drop=True)
    return df.iloc[0], df


def generate_patient_home_for_hospital(hospital_id, activity_type, hospitals_df):
    target_df = hospitals_df[hospitals_df["hospital_id"] == hospital_id]
    if len(target_df) != 1:
        raise ValueError(f"Unknown hospital_id: {hospital_id}")

    target = target_df.iloc[0]
    radius_km = CATCHMENT_RADIUS_BY_ACTIVITY_KM.get(activity_type, 12)
    best_candidate = None
    best_penalty = 10**9

    for _ in range(MAX_LOCATION_ATTEMPTS):
        lat, lon = random_point_around(
            float(target["lat"]),
            float(target["lon"]),
            radius_km,
        )
        closest, all_distances = closest_hospital_for_point(lat, lon, hospitals_df)
        assigned_distance = float(
            all_distances.loc[
                all_distances["hospital_id"] == hospital_id,
                "distance_km",
            ].iloc[0]
        )
        closest_distance = float(closest["distance_km"])
        closest_hospital_id = str(closest["hospital_id"])
        assigned_is_coherent = (
            assigned_distance <= closest_distance + NEAREST_HOSPITAL_TOLERANCE_KM
        )
        penalty = assigned_distance - closest_distance
        candidate_info = {
            "lat": lat,
            "lon": lon,
            "assigned_distance_km": assigned_distance,
            "closest_hospital_id": closest_hospital_id,
            "closest_distance_km": closest_distance,
            "assigned_hospital_is_coherent": int(assigned_is_coherent),
            "generation_method": "catchment_near_assigned_hospital",
            "catchment_radius_km": radius_km,
        }
        if assigned_is_coherent:
            return lat, lon, candidate_info
        if penalty < best_penalty:
            best_penalty = penalty
            best_candidate = candidate_info

    if STRICT_PATIENT_CATCHMENT:
        raise RuntimeError(
            f"Could not generate coherent location for hospital={hospital_id}, "
            f"activity={activity_type}. Check coordinates or catchment settings."
        )
    return best_candidate["lat"], best_candidate["lon"], best_candidate


def random_vehicle_start_point():
    return random_point_around(46.0500, 4.0700, radius_km=10)


# ------------------------------------------------------------
# Required helpers from your original code
# ------------------------------------------------------------
_REQUIRED_HELPERS = [
    "minute_to_datetime",
    "datetime_to_minute",
    "hhmm_to_min",
    "min_to_hhmm",
    "road_distance_est_km",
    "split_capacity_by_weights",
    "add_activity_schedule",
    "scaled_capacity",
    "compute_max_ride_time_min",
    "compatible",
    "generate_patient_home_for_hospital",
    "random_vehicle_start_point",
]
_missing = [name for name in _REQUIRED_HELPERS if name not in globals()]
if _missing:
    raise RuntimeError(
        "Run the imports/global parameters/helper-functions cell from your original code first. "
        f"Missing: {_missing}"
    )

# ------------------------------------------------------------
# Global parameters
# ------------------------------------------------------------
SEED = globals().get("SEED", 42)
SERVICE_DATE = globals().get("SERVICE_DATE", "2026-05-18")

TRANSPORT_START_MIN = globals().get("TRANSPORT_START_MIN", 6 * 60)
TRANSPORT_END_MIN = globals().get("TRANSPORT_END_MIN", 18 * 60)
APPOINTMENT_START_MIN = globals().get("APPOINTMENT_START_MIN", 8 * 60)
APPOINTMENT_END_MIN = globals().get("APPOINTMENT_END_MIN", 18 * 60)

SERVICE_PICKUP_MIN = globals().get("SERVICE_PICKUP_MIN", 5)
SERVICE_DROPOFF_MIN = globals().get("SERVICE_DROPOFF_MIN", 5)
EXTERNAL_TRANSPORT_SHARE = globals().get("EXTERNAL_TRANSPORT_SHARE", 0.15)
ROAD_FACTOR = globals().get("ROAD_FACTOR", 1.25)

OMEGA_MAX_EARLY_ARRIVAL_MIN = globals().get("OMEGA_MAX_EARLY_ARRIVAL_MIN", 45)
PSI_MAX_RETURN_DELAY_MIN = globals().get("PSI_MAX_RETURN_DELAY_MIN", 35)
LAMBDA_RIDE_TIME_WEIGHT = globals().get("LAMBDA_RIDE_TIME_WEIGHT", 1.0)
BIG_M = globals().get("BIG_M", 100000)
DEFAULT_SPEED_KMH = globals().get("DEFAULT_SPEED_KMH", 42)
DEFAULT_COST_PER_KM = globals().get("DEFAULT_COST_PER_KM", 1.25)

ROOT_DIR = "/content/darp_roanne_instances"
ALL_OUTPUT_ZIP = "/content/darp_roanne_all_instances.zip"

# Set True only if your colleague really needs t_ijvk/c_ijvk for every vehicle id.
# Warning: for L instances this can create hundreds of millions of rows across all instances.
EXPORT_VEHICLE_ID_MATRIX = False

# ------------------------------------------------------------
# Instance grid from your table
# ------------------------------------------------------------
def build_instance_grid():
    rows = []
    levels = [
        ("E", [20, 50, 75], [5, 10], [1, 2]),
        ("S", [100, 200, 250], [15, 20], [2, 3]),
        ("M", [300, 350, 400, 450], [20, 30, 40], [3]),
        ("L", [475, 550, 650, 750], [50, 60, 70], [3]),
    ]
    for size, patient_values, vehicle_values, hospital_values in levels:
        for n_patients in patient_values:
            for n_vehicles in vehicle_values:
                for n_hospitals in hospital_values:
                    rows.append({
                        "category": f"{size}/{n_patients}/{n_vehicles}/{n_hospitals}",
                        "size": size,
                        "n_patients": int(n_patients),
                        "n_vehicles": int(n_vehicles),
                        "n_hospitals": int(n_hospitals),
                    })
    return pd.DataFrame(rows)

INSTANCE_GRID = build_instance_grid()

# ------------------------------------------------------------
# Activity and vehicle parameters
# ------------------------------------------------------------
activity_priority = {
    "dialysis": {
        "priority_code": "P1",
        "priority_class": "high",
        "waiting_time_after_treatment_min": 15,
    },
    "oncology": {
        "priority_code": "P1",
        "priority_class": "high",
        "waiting_time_after_treatment_min": 15,
    },
    "other": {
        "priority_code": "P2",
        "priority_class": "medium",
        "waiting_time_after_treatment_min": 25,
    },
    "external": {
        "priority_code": "P3",
        "priority_class": "low",
        "waiting_time_after_treatment_min": 35,
    },
}

theta_by_priority = {
    "low": 1.0,
    "medium": 1.5,
    "high": 2.0,
}

activity_duration_rules = {
    ("H1", "dialysis"): (240, 240),
    ("H2", "dialysis"): (210, 270),
    ("H3", "dialysis"): (210, 270),
    ("H1", "oncology"): (150, 180),
    ("H2", "oncology"): (60, 360),
    ("H1", "other"): (60, 120),
    ("H2", "other"): (25, 40),
    ("H1", "external"): (20, 30),
}

same_day_return_prob_by_activity = {
    "dialysis": 0.99,
    "oncology": 0.99,
    "other": 0.95,
    "external": 0.90,
}

requirement_probs_by_activity = {
    "dialysis": (
        ["vsl_required", "prescription_medicale"],
        [0.70, 0.30],
    ),
    "oncology": (
        ["vsl_required", "prescription_medicale", "standard_assis"],
        [0.50, 0.35, 0.15],
    ),
    "other": (
        ["standard_assis", "prescription_medicale", "vsl_required", "premium"],
        [0.60, 0.25, 0.10, 0.05],
    ),
    "external": (
        ["standard_assis", "prescription_medicale", "premium"],
        [0.70, 0.25, 0.05],
    ),
}

SLOT_SPREAD_MINUTES_BY_ACTIVITY = {
    "dialysis": 60,
    "oncology": 60,
    "other": 90,
    "external": 120,
}

APPOINTMENT_STEP_MIN = 5

capacity_by_type = {
    "taxi_conventionne": 3,
    "vsl": 3,
    "vtc_premium": 3,
}

speed_by_type_kmh = {
    "taxi_conventionne": 42,
    "vsl": 40,
    "vtc_premium": 45,
}

cost_by_type_km = {
    "taxi_conventionne": 1.25,
    "vsl": 1.35,
    "vtc_premium": 1.70,
}

# ------------------------------------------------------------
# Builders
# ------------------------------------------------------------
def safe_instance_name(category):
    return str(category).replace("/", "_")


def sample_duration_min_instance(hospital_id, activity_type):
    lo, hi = activity_duration_rules.get((hospital_id, activity_type), (30, 60))
    if int(lo) == int(hi):
        return int(lo)
    return int(random.randrange(int(lo), int(hi) + 1, 5))


def sample_requirement_for_activity_instance(activity_type):
    values, probs = requirement_probs_by_activity[activity_type]
    return str(np.random.choice(values, p=probs))


def make_vehicle_type_plan(n_vehicles):
    vehicle_types = ["taxi_conventionne", "vsl", "vtc_premium"]
    weights = [0.50, 0.35, 0.15]
    counts = split_capacity_by_weights(int(n_vehicles), weights)
    plan = []
    for vehicle_type, count in zip(vehicle_types, counts):
        plan.extend([vehicle_type] * int(count))
    if len(plan) != int(n_vehicles):
        raise RuntimeError("Vehicle type plan length mismatch.")
    return plan


def build_hospitals(n_hospitals):
    hospitals_pool = pd.DataFrame([
        {
            "hospital_id": "H1",
            "hospital_name": "Centre Hospitalier de Roanne",
            "city": "Roanne",
            "hospital_type": "public_hospital",
            "lat": 46.0533,
            "lon": 4.0727,
            "source_quality": "observed_user_table",
        },
        {
            "hospital_id": "H2",
            "hospital_name": "Clinique du Renaison",
            "city": "Roanne",
            "hospital_type": "private_clinic",
            "lat": 46.0457,
            "lon": 4.0688,
            "source_quality": "partial_user_table_assumption_needed",
        },
        {
            "hospital_id": "H3",
            "hospital_name": "Centre de dialyse independant - Roanne",
            "city": "Roanne",
            "hospital_type": "dialysis_center",
            "lat": 46.0500,
            "lon": 4.0700,
            "source_quality": "assumption_replace_with_real_center",
        },
    ])
    if int(n_hospitals) not in [1, 2, 3]:
        raise ValueError("n_hospitals must be 1, 2, or 3.")
    return hospitals_pool.iloc[:int(n_hospitals)].copy().reset_index(drop=True)


def build_base_activity_schedule(hospitals_df):
    schedule_rows = []
    hospital_ids = set(hospitals_df["hospital_id"].astype(str))

    if "H1" in hospital_ids:
        add_activity_schedule(
            schedule_rows,
            hospital_id="H1",
            activity_type="dialysis",
            daily_capacity=53,
            slot_times=["08:00", "13:00", "17:00"],
            slot_capacities=[26, 23, 4],
            source_quality="observed_user_table",
            source_note="CH Roanne dialysis: 53/day. 26 morning, 23 noon, 4 evening.",
        )
        add_activity_schedule(
            schedule_rows,
            hospital_id="H1",
            activity_type="oncology",
            daily_capacity=30,
            slot_times=["08:00", "11:00", "14:00"],
            slot_weights=[0.55, 0.35, 0.10],
            source_quality="observed_user_table",
            source_note="CH Roanne chemotherapy: 30/day.",
        )
        add_activity_schedule(
            schedule_rows,
            hospital_id="H1",
            activity_type="other",
            daily_capacity=49,
            slot_times=["08:30", "10:30", "14:00", "16:00"],
            slot_weights=[0.40, 0.30, 0.20, 0.10],
            source_quality="observed_user_table",
            source_note="CH Roanne other: 49/day.",
        )
        h1_external_daily = scaled_capacity(75, EXTERNAL_TRANSPORT_SHARE)
        add_activity_schedule(
            schedule_rows,
            hospital_id="H1",
            activity_type="external",
            daily_capacity=h1_external_daily,
            slot_times=[
                "08:30", "09:30", "10:30", "11:30", "12:30",
                "13:30", "14:30", "15:30", "16:30", "17:30",
            ],
            slot_weights=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
            source_quality="computed_from_user_table",
            source_note="External visits: 75/day * 15% using fleet.",
        )

    if "H2" in hospital_ids:
        add_activity_schedule(
            schedule_rows,
            hospital_id="H2",
            activity_type="dialysis",
            daily_capacity=30,
            slot_times=["08:00", "13:00", "17:00"],
            slot_weights=[0.50, 0.45, 0.05],
            source_quality="assumption_to_validate",
            source_note="Renaison dialysis: assumed 30/day.",
        )
        add_activity_schedule(
            schedule_rows,
            hospital_id="H2",
            activity_type="oncology",
            daily_capacity=16,
            slot_times=["08:00", "13:00", "15:00"],
            slot_weights=[0.55, 0.35, 0.10],
            source_quality="assumption_to_validate",
            source_note="Renaison oncology: assumed 16/day.",
        )
        add_activity_schedule(
            schedule_rows,
            hospital_id="H2",
            activity_type="other",
            daily_capacity=36,
            slot_times=["08:30", "10:30", "14:00", "16:30"],
            slot_weights=[0.35, 0.25, 0.25, 0.15],
            source_quality="assumption_to_validate",
            source_note="Renaison other: assumed 36/day.",
        )

    if "H3" in hospital_ids:
        add_activity_schedule(
            schedule_rows,
            hospital_id="H3",
            activity_type="dialysis",
            daily_capacity=24,
            slot_times=["08:00", "13:00", "17:00"],
            slot_weights=[0.50, 0.45, 0.05],
            source_quality="assumption_to_validate",
            source_note="Third dialysis center: assumed 24/day.",
        )

    return pd.DataFrame(schedule_rows)


def scale_schedule_to_patient_count(schedule_df, n_patients):
    """Scale one instance while preserving the hospital/activity hierarchy.

    Allocation is performed in three exact integer stages:
      1. total patients -> hospitals, using each available hospital's reference
         capacity as its weight;
      2. hospital patients -> activities, using the reference activity mix of
         that hospital;
      3. activity patients -> appointment slots, using the reference slot mix.

    The largest-remainder method in ``split_capacity_by_weights`` guarantees
    that every stage sums exactly to the requested instance patient count.
    If an instance contains only H1 or H1/H2, hospital shares are renormalized
    over the hospitals that are actually present in that instance.
    """
    out = schedule_df.copy().reset_index(drop=True)
    n_patients = int(n_patients)
    if n_patients < 0:
        raise ValueError("n_patients must be non-negative.")
    if out.empty:
        if n_patients == 0:
            return out
        raise ValueError("Cannot allocate patients without an activity schedule.")

    # Preserve the unscaled values for traceability in every exported instance.
    out["reference_daily_capacity"] = out["daily_capacity"].astype(int)
    out["reference_slot_capacity"] = out["slot_capacity"].astype(int)

    hospital_reference = (
        out.groupby("hospital_id", sort=False)["reference_slot_capacity"]
        .sum()
        .astype(int)
    )
    hospital_counts = split_capacity_by_weights(
        n_patients,
        hospital_reference.to_numpy(dtype=float),
    )
    hospital_target = dict(zip(hospital_reference.index, hospital_counts))
    reference_total = int(hospital_reference.sum())

    out["hospital_reference_capacity"] = out["hospital_id"].map(
        hospital_reference.to_dict()
    ).astype(int)
    out["hospital_reference_share"] = (
        out["hospital_reference_capacity"] / float(reference_total)
    )
    out["hospital_target_patients"] = out["hospital_id"].map(hospital_target).astype(int)
    out["hospital_actual_share"] = (
        out["hospital_target_patients"] / float(n_patients)
        if n_patients > 0
        else 0.0
    )

    # Allocate each hospital's patients to its own specialties/activities.
    out["activity_reference_capacity"] = 0
    out["activity_reference_share_within_hospital"] = 0.0
    out["activity_target_patients"] = 0
    out["activity_actual_share_within_hospital"] = 0.0

    for hospital_id, hospital_rows in out.groupby("hospital_id", sort=False):
        hospital_mask = out["hospital_id"] == hospital_id
        activity_reference = (
            hospital_rows.groupby("activity_type", sort=False)["reference_slot_capacity"]
            .sum()
            .astype(int)
        )
        activity_counts = split_capacity_by_weights(
            hospital_target[hospital_id],
            activity_reference.to_numpy(dtype=float),
        )

        for activity_type, activity_count in zip(activity_reference.index, activity_counts):
            activity_mask = hospital_mask & (out["activity_type"] == activity_type)
            slot_reference = out.loc[activity_mask, "reference_slot_capacity"].to_numpy(dtype=float)
            slot_counts = split_capacity_by_weights(activity_count, slot_reference)

            out.loc[activity_mask, "slot_capacity"] = slot_counts
            out.loc[activity_mask, "daily_capacity"] = int(activity_count)
            out.loc[activity_mask, "activity_reference_capacity"] = int(
                activity_reference[activity_type]
            )
            out.loc[activity_mask, "activity_reference_share_within_hospital"] = (
                float(activity_reference[activity_type])
                / float(hospital_reference[hospital_id])
            )
            out.loc[activity_mask, "activity_target_patients"] = int(activity_count)
            out.loc[activity_mask, "activity_actual_share_within_hospital"] = (
                float(activity_count) / float(hospital_target[hospital_id])
                if hospital_target[hospital_id] > 0
                else 0.0
            )

    integer_columns = [
        "slot_capacity",
        "daily_capacity",
        "activity_reference_capacity",
        "activity_target_patients",
    ]
    out[integer_columns] = out[integer_columns].astype(int)

    if int(out["slot_capacity"].sum()) != n_patients:
        raise RuntimeError("Hierarchical patient allocation does not match instance total.")

    hospital_check = out.groupby("hospital_id")["slot_capacity"].sum().astype(int)
    for hospital_id, expected_count in hospital_target.items():
        if int(hospital_check.get(hospital_id, 0)) != int(expected_count):
            raise RuntimeError(
                f"Hospital allocation mismatch for {hospital_id}: "
                f"expected {expected_count}, got {hospital_check.get(hospital_id, 0)}."
            )

    return out


def build_instance_distribution_tables(hospital_activity_schedule, n_patients):
    """Return explicit hospital and hospital-specialty distributions."""
    n_patients = int(n_patients)

    hospital_distribution = (
        hospital_activity_schedule
        .groupby("hospital_id", sort=False, as_index=False)
        .agg(
            reference_capacity=("reference_slot_capacity", "sum"),
            target_patients=("slot_capacity", "sum"),
        )
    )
    hospital_distribution["reference_share"] = (
        hospital_distribution["reference_capacity"]
        / hospital_distribution["reference_capacity"].sum()
    )
    hospital_distribution["actual_share"] = (
        hospital_distribution["target_patients"] / float(n_patients)
        if n_patients > 0
        else 0.0
    )
    hospital_distribution["reference_share_pct"] = (
        100.0 * hospital_distribution["reference_share"]
    ).round(4)
    hospital_distribution["actual_share_pct"] = (
        100.0 * hospital_distribution["actual_share"]
    ).round(4)

    hospital_specialty_distribution = (
        hospital_activity_schedule
        .groupby(["hospital_id", "activity_type"], sort=False, as_index=False)
        .agg(
            reference_capacity=("reference_slot_capacity", "sum"),
            target_patients=("slot_capacity", "sum"),
        )
    )
    reference_by_hospital = hospital_specialty_distribution.groupby("hospital_id")[
        "reference_capacity"
    ].transform("sum")
    target_by_hospital = hospital_specialty_distribution.groupby("hospital_id")[
        "target_patients"
    ].transform("sum")
    hospital_specialty_distribution["reference_share_within_hospital"] = (
        hospital_specialty_distribution["reference_capacity"] / reference_by_hospital
    )
    hospital_specialty_distribution["actual_share_within_hospital"] = np.where(
        target_by_hospital > 0,
        hospital_specialty_distribution["target_patients"] / target_by_hospital,
        0.0,
    )
    hospital_specialty_distribution["actual_share_of_instance"] = (
        hospital_specialty_distribution["target_patients"] / float(n_patients)
        if n_patients > 0
        else 0.0
    )
    for column in [
        "reference_share_within_hospital",
        "actual_share_within_hospital",
        "actual_share_of_instance",
    ]:
        hospital_specialty_distribution[f"{column}_pct"] = (
            100.0 * hospital_specialty_distribution[column]
        ).round(4)

    return hospital_distribution, hospital_specialty_distribution


def make_distance_matrix_points_vectorized(points_df):
    ids = points_df["point_id"].astype(str).to_numpy()
    types = points_df["point_type"].astype(str).to_numpy()
    lat = np.radians(points_df["lat"].astype(float).to_numpy())
    lon = np.radians(points_df["lon"].astype(float).to_numpy())
    n = len(points_df)

    dlat = lat[None, :] - lat[:, None]
    dlon = lon[None, :] - lon[:, None]
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2.0) ** 2
    )
    km = 6371.0 * 2.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1))) * ROAD_FACTOR
    km = np.round(km, 5)

    return pd.DataFrame({
        "origin_point_id": np.repeat(ids, n),
        "destination_point_id": np.tile(ids, n),
        "origin_point_type": np.repeat(types, n),
        "destination_point_type": np.tile(types, n),
        "distance_km": km.ravel(),
        "distance_m": np.round(km.ravel() * 1000.0, 2),
        "distance_source": "haversine_x_road_factor",
    })


def export_travel_matrix_by_vehicle_stream(distance_matrix_points, vehicles, output_path):
    base = distance_matrix_points[["origin_point_id", "destination_point_id", "distance_km"]].copy()
    first = True
    with gzip.open(output_path, "wt", newline="", encoding="utf-8") as f:
        for _, v in vehicles.iterrows():
            tmp = base.copy()
            tmp["vehicle_id"] = v["vehicle_id"]
            tmp["vehicle_type"] = v["vehicle_type"]
            tmp["t_ijvk"] = (tmp["distance_km"] / float(v["speed_kmh"]) * 60.0).round(2)
            tmp["c_ijvk"] = (tmp["distance_km"] * float(v["cost_per_km"])).round(2)
            tmp.to_csv(f, index=False, header=first)
            first = False

# ------------------------------------------------------------
# Main single-instance generator
# ------------------------------------------------------------
def generate_one_instance(cfg, instance_number):
    seed = int(SEED) + int(instance_number)
    random.seed(seed)
    np.random.seed(seed)

    instance_name = safe_instance_name(cfg["category"])
    base_dir = os.path.join(ROOT_DIR, instance_name)
    csv_dir = os.path.join(base_dir, "csv")
    matrix_dir = os.path.join(base_dir, "matrix")
    output_excel = os.path.join(base_dir, f"{instance_name}.xlsx")
    output_distance_matrix_excel = os.path.join(base_dir, "distance_matrix_only.xlsx")

    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    os.makedirs(csv_dir, exist_ok=True)
    os.makedirs(matrix_dir, exist_ok=True)

    n_patients_target = int(cfg["n_patients"])
    n_vehicles_target = int(cfg["n_vehicles"])
    n_hospitals_target = int(cfg["n_hospitals"])

    hospitals = build_hospitals(n_hospitals_target)
    hospital_activity_schedule = build_base_activity_schedule(hospitals)
    hospital_activity_schedule = scale_schedule_to_patient_count(
        hospital_activity_schedule,
        n_patients_target,
    )
    hospital_distribution, hospital_specialty_distribution = (
        build_instance_distribution_tables(
            hospital_activity_schedule,
            n_patients_target,
        )
    )
    hospital_distribution = hospital_distribution.merge(
        hospitals[["hospital_id", "hospital_name"]],
        on="hospital_id",
        how="left",
    )
    hospital_specialty_distribution = hospital_specialty_distribution.merge(
        hospitals[["hospital_id", "hospital_name"]],
        on="hospital_id",
        how="left",
    )

    daily_capacity_check = (
        hospital_activity_schedule
        .groupby(["hospital_id", "activity_type"], as_index=False)
        .agg(
            daily_capacity=("daily_capacity", "first"),
            total_slot_capacity=("slot_capacity", "sum"),
        )
    )
    capacity_mismatch = daily_capacity_check[
        daily_capacity_check["daily_capacity"] != daily_capacity_check["total_slot_capacity"]
    ]
    if len(capacity_mismatch) > 0:
        raise RuntimeError(f"Capacity mismatch in {instance_name}")

    vehicle_type_plan = make_vehicle_type_plan(n_vehicles_target)
    vehicle_rows = []
    for idx, vehicle_type in enumerate(vehicle_type_plan, start=1):
        lat, lon = random_vehicle_start_point()
        vehicle_rows.append({
            "vehicle_id": f"V{idx}",
            "vehicle_type": vehicle_type,
            "capacity_Qk": capacity_by_type[vehicle_type],
            "speed_kmh": speed_by_type_kmh[vehicle_type],
            "cost_per_km": cost_by_type_km[vehicle_type],
            "start_point_id": f"VS_V{idx}",
            "start_lat": lat,
            "start_lon": lon,
            "route_type": "open_route",
        })
    vehicles = pd.DataFrame(vehicle_rows)

    # Appointment plan
    appointment_rows = []
    appointment_idx = 1
    for _, slot in hospital_activity_schedule.iterrows():
        hospital_id = slot["hospital_id"]
        activity_type = slot["activity_type"]
        spread = SLOT_SPREAD_MINUTES_BY_ACTIVITY.get(activity_type, 60)
        possible_offsets = [
            offset
            for offset in range(0, spread, APPOINTMENT_STEP_MIN)
            if int(slot["slot_start_min"]) + offset <= APPOINTMENT_END_MIN
        ]
        if len(possible_offsets) == 0:
            possible_offsets = [0]

        for _ in range(int(slot["slot_capacity"])):
            appointment_min = int(slot["slot_start_min"] + random.choice(possible_offsets))
            duration_min = sample_duration_min_instance(hospital_id, activity_type)
            priority_info = activity_priority[activity_type]
            appointment_rows.append({
                "appointment_id": f"A{appointment_idx:04d}",
                "hospital_id": hospital_id,
                "activity_type": activity_type,
                "slot_time": slot["slot_time"],
                "slot_start_min": int(slot["slot_start_min"]),
                "appointment_minute": appointment_min,
                "appointment_time_hhmm": min_to_hhmm(appointment_min),
                "treatment_duration_min": int(duration_min),
                "priority_code": priority_info["priority_code"],
                "priority_class": priority_info["priority_class"],
                "waiting_time_after_treatment_min": int(priority_info["waiting_time_after_treatment_min"]),
                "source_quality": slot["source_quality"],
                "source_note": slot["source_note"],
            })
            appointment_idx += 1

    appointment_plan = pd.DataFrame(appointment_rows)
    if len(appointment_plan) != n_patients_target:
        raise RuntimeError(
            f"{instance_name}: generated {len(appointment_plan)} appointments, expected {n_patients_target}."
        )
    appointment_plan = appointment_plan.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Patients, points, requests, model nodes
    patient_rows = []
    point_rows = []
    request_rows = []
    model_node_rows = []
    future_return_rows = []

    hospital_lookup = hospitals.set_index("hospital_id").to_dict("index")

    point_rows.append({
        "point_id": "D0",
        "point_type": "depot",
        "lat": 46.0500,
        "lon": 4.0700,
        "patient_id": None,
        "hospital_id": None,
        "vehicle_id": None,
        "label": "Depot Roanne",
    })

    for _, h in hospitals.iterrows():
        point_rows.append({
            "point_id": h["hospital_id"],
            "point_type": "hospital",
            "lat": h["lat"],
            "lon": h["lon"],
            "patient_id": None,
            "hospital_id": h["hospital_id"],
            "vehicle_id": None,
            "label": h["hospital_name"],
        })

    for _, v in vehicles.iterrows():
        point_rows.append({
            "point_id": v["start_point_id"],
            "point_type": "vehicle_start",
            "lat": v["start_lat"],
            "lon": v["start_lon"],
            "patient_id": None,
            "hospital_id": None,
            "vehicle_id": v["vehicle_id"],
            "label": "vehicle_start",
        })

    patient_idx = 1
    request_idx = 1
    model_node_idx = 1

    for _, appt in appointment_plan.iterrows():
        patient_id = f"P{patient_idx}"
        home_point_id = f"HOME_{patient_id}"
        hospital_id = appt["hospital_id"]
        activity_type = appt["activity_type"]
        hospital = hospital_lookup[hospital_id]
        hospital_lat = float(hospital["lat"])
        hospital_lon = float(hospital["lon"])

        home_lat, home_lon, location_info = generate_patient_home_for_hospital(
            hospital_id=hospital_id,
            activity_type=activity_type,
            hospitals_df=hospitals,
        )

        priority_class = appt["priority_class"]
        theta = theta_by_priority[priority_class]
        requirement = sample_requirement_for_activity_instance(activity_type)

        distance_km = road_distance_est_km(home_lat, home_lon, hospital_lat, hospital_lon)
        direct_time_min = max(5.0, distance_km / DEFAULT_SPEED_KMH * 60.0)
        max_ride_min = compute_max_ride_time_min(direct_time_min, priority_class)

        appointment_min = int(appt["appointment_minute"])
        appointment_dt = minute_to_datetime(SERVICE_DATE, appointment_min)
        treatment_duration_min = int(appt["treatment_duration_min"])
        return_ready_dt = appointment_dt + pd.Timedelta(minutes=treatment_duration_min)
        return_ready_min = datetime_to_minute(return_ready_dt, SERVICE_DATE)

        has_return_today = bool(random.random() < same_day_return_prob_by_activity[activity_type])

        point_rows.append({
            "point_id": home_point_id,
            "point_type": "patient_home",
            "lat": home_lat,
            "lon": home_lon,
            "patient_id": patient_id,
            "hospital_id": hospital_id,
            "vehicle_id": None,
            "label": "patient_home",
        })

        patient_rows.append({
            "patient_id": patient_id,
            "home_point_id": home_point_id,
            "home_lat": home_lat,
            "home_lon": home_lon,
            "hospital_id": hospital_id,
            "activity_type": activity_type,
            "appointment_id": appt["appointment_id"],
            "slot_time": appt["slot_time"],
            "priority_code": appt["priority_code"],
            "priority_class": priority_class,
            "theta_p": theta,
            "requirement": requirement,
            "appointment_datetime": appointment_dt,
            "appointment_minute": appointment_min,
            "appointment_time_hhmm": min_to_hhmm(appointment_min),
            "hospital_stay_min": treatment_duration_min,
            "return_ready_datetime": return_ready_dt,
            "return_ready_minute": return_ready_min,
            "return_ready_time_hhmm": min_to_hhmm(return_ready_min),
            "waiting_time_after_treatment_min": int(appt["waiting_time_after_treatment_min"]),
            "return_same_day": int(has_return_today),
            "max_ride_time_min": int(max_ride_min),
            "direct_distance_est_km": round(distance_km, 3),
            "direct_travel_time_est_min": round(direct_time_min, 2),
            "distance_to_assigned_hospital_km": round(location_info["assigned_distance_km"], 3),
            "closest_hospital_id": location_info["closest_hospital_id"],
            "closest_hospital_distance_km": round(location_info["closest_distance_km"], 3),
            "assigned_hospital_is_coherent": int(location_info["assigned_hospital_is_coherent"]),
            "home_generation_method": location_info["generation_method"],
            "catchment_radius_km": location_info["catchment_radius_km"],
            "source_quality": appt["source_quality"],
            "source_note": appt["source_note"],
        })

        # Outbound request
        request_id = f"R{request_idx:04d}"
        pickup_model_node_label = f"PU_{request_id}"
        dropoff_model_node_label = f"DO_{request_id}"
        earliest_pickup_min = max(TRANSPORT_START_MIN, appointment_min - int(max_ride_min) - 30)
        latest_pickup_min = max(TRANSPORT_START_MIN, appointment_min - int(math.ceil(direct_time_min)))

        request_rows.append({
            "request_id": request_id,
            "patient_id": patient_id,
            "trip_type": "outbound",
            "direction": "patient_to_hospital",
            "origin_point_id": home_point_id,
            "destination_point_id": hospital_id,
            "hospital_id": hospital_id,
            "activity_type": activity_type,
            "priority_code": appt["priority_code"],
            "priority_class": priority_class,
            "theta_p": theta,
            "requirement": requirement,
            "service_pickup_min": SERVICE_PICKUP_MIN,
            "service_dropoff_min": SERVICE_DROPOFF_MIN,
            "q_pickup": 1,
            "q_dropoff": -1,
            "earliest_pickup_min": int(earliest_pickup_min),
            "latest_pickup_min": int(latest_pickup_min),
            "latest_dropoff_min": int(appointment_min),
            "appointment_minute_a_p": int(appointment_min),
            "max_ride_time_L_p_min": int(max_ride_min),
            "in_planning_horizon": 1,
            "paired_request_id": None,
            "pickup_model_node_label": pickup_model_node_label,
            "dropoff_model_node_label": dropoff_model_node_label,
        })

        model_node_rows.append({
            "model_node_id": model_node_idx,
            "model_node_label": pickup_model_node_label,
            "request_id": request_id,
            "patient_id": patient_id,
            "node_role": "pickup",
            "physical_point_id": home_point_id,
            "lat": home_lat,
            "lon": home_lon,
            "service_time_min": SERVICE_PICKUP_MIN,
            "q_i": 1,
            "time_window_start_min": int(earliest_pickup_min),
            "time_window_end_min": int(latest_pickup_min),
        })
        model_node_idx += 1

        model_node_rows.append({
            "model_node_id": model_node_idx,
            "model_node_label": dropoff_model_node_label,
            "request_id": request_id,
            "patient_id": patient_id,
            "node_role": "dropoff",
            "physical_point_id": hospital_id,
            "lat": hospital_lat,
            "lon": hospital_lon,
            "service_time_min": SERVICE_DROPOFF_MIN,
            "q_i": -1,
            "time_window_start_min": max(TRANSPORT_START_MIN, int(appointment_min) - OMEGA_MAX_EARLY_ARRIVAL_MIN),
            "time_window_end_min": int(appointment_min),
        })
        model_node_idx += 1
        request_idx += 1

        # Return request
        if has_return_today:
            wait_after_treatment = int(appt["waiting_time_after_treatment_min"])
            return_pickup_min = int(return_ready_min)
            return_latest_pickup_min = int(
                return_pickup_min + min(wait_after_treatment, PSI_MAX_RETURN_DELAY_MIN)
            )
            return_latest_dropoff_min = int(return_latest_pickup_min + max_ride_min + 15)

            if return_pickup_min <= TRANSPORT_END_MIN and return_latest_dropoff_min <= TRANSPORT_END_MIN:
                return_request_id = f"R{request_idx:04d}"
                pickup_model_node_label_r = f"PU_{return_request_id}"
                dropoff_model_node_label_r = f"DO_{return_request_id}"

                request_rows.append({
                    "request_id": return_request_id,
                    "patient_id": patient_id,
                    "trip_type": "return",
                    "direction": "hospital_to_patient",
                    "origin_point_id": hospital_id,
                    "destination_point_id": home_point_id,
                    "hospital_id": hospital_id,
                    "activity_type": activity_type,
                    "priority_code": appt["priority_code"],
                    "priority_class": priority_class,
                    "theta_p": theta,
                    "requirement": requirement,
                    "service_pickup_min": SERVICE_DROPOFF_MIN,
                    "service_dropoff_min": SERVICE_PICKUP_MIN,
                    "q_pickup": 1,
                    "q_dropoff": -1,
                    "earliest_pickup_min": int(return_pickup_min),
                    "latest_pickup_min": int(return_latest_pickup_min),
                    "latest_dropoff_min": int(return_latest_dropoff_min),
                    "appointment_minute_a_p": int(return_pickup_min),
                    "max_ride_time_L_p_min": int(max_ride_min),
                    "in_planning_horizon": 1,
                    "paired_request_id": request_id,
                    "pickup_model_node_label": pickup_model_node_label_r,
                    "dropoff_model_node_label": dropoff_model_node_label_r,
                })
                request_rows[-2]["paired_request_id"] = return_request_id

                model_node_rows.append({
                    "model_node_id": model_node_idx,
                    "model_node_label": pickup_model_node_label_r,
                    "request_id": return_request_id,
                    "patient_id": patient_id,
                    "node_role": "pickup",
                    "physical_point_id": hospital_id,
                    "lat": hospital_lat,
                    "lon": hospital_lon,
                    "service_time_min": SERVICE_DROPOFF_MIN,
                    "q_i": 1,
                    "time_window_start_min": int(return_pickup_min),
                    "time_window_end_min": int(return_latest_pickup_min),
                })
                model_node_idx += 1

                model_node_rows.append({
                    "model_node_id": model_node_idx,
                    "model_node_label": dropoff_model_node_label_r,
                    "request_id": return_request_id,
                    "patient_id": patient_id,
                    "node_role": "dropoff",
                    "physical_point_id": home_point_id,
                    "lat": home_lat,
                    "lon": home_lon,
                    "service_time_min": SERVICE_PICKUP_MIN,
                    "q_i": -1,
                    "time_window_start_min": int(return_pickup_min),
                    "time_window_end_min": int(return_latest_dropoff_min),
                })
                model_node_idx += 1
                request_idx += 1
            else:
                future_return_rows.append({
                    "patient_id": patient_id,
                    "hospital_id": hospital_id,
                    "activity_type": activity_type,
                    "origin_point_id": hospital_id,
                    "destination_point_id": home_point_id,
                    "return_ready_datetime": return_ready_dt,
                    "return_ready_minute_relative_to_service_date": int(return_ready_min),
                    "reason_not_in_day_dataset": "return_after_transport_horizon",
                })
        else:
            future_return_rows.append({
                "patient_id": patient_id,
                "hospital_id": hospital_id,
                "activity_type": activity_type,
                "origin_point_id": hospital_id,
                "destination_point_id": home_point_id,
                "return_ready_datetime": return_ready_dt,
                "return_ready_minute_relative_to_service_date": int(return_ready_min),
                "reason_not_in_day_dataset": "return_not_generated_by_probability",
            })

        patient_idx += 1

    patients = pd.DataFrame(patient_rows)
    requests = pd.DataFrame(request_rows)
    future_returns = pd.DataFrame(future_return_rows)
    points = pd.DataFrame(point_rows).drop_duplicates(subset=["point_id"]).reset_index(drop=True)
    model_nodes = pd.DataFrame(model_node_rows)

    datetime_columns = {
        "earliest_pickup_min": "earliest_pickup_datetime",
        "latest_pickup_min": "latest_pickup_datetime",
        "latest_dropoff_min": "latest_dropoff_datetime",
        "appointment_minute_a_p": "appointment_datetime_a_p",
    }
    for source_col, target_col in datetime_columns.items():
        requests[target_col] = requests[source_col].apply(
            lambda m: minute_to_datetime(SERVICE_DATE, int(m))
        )
    requests["earliest_pickup_hhmm"] = requests["earliest_pickup_min"].apply(min_to_hhmm)
    requests["latest_pickup_hhmm"] = requests["latest_pickup_min"].apply(min_to_hhmm)
    requests["latest_dropoff_hhmm"] = requests["latest_dropoff_min"].apply(min_to_hhmm)

    # Compatibility
    compatibility_rows = []
    for _, r in requests.iterrows():
        for _, v in vehicles.iterrows():
            compatibility_rows.append({
                "request_id": r["request_id"],
                "patient_id": r["patient_id"],
                "trip_type": r["trip_type"],
                "hospital_id": r["hospital_id"],
                "activity_type": r["activity_type"],
                "vehicle_id": v["vehicle_id"],
                "vehicle_type": v["vehicle_type"],
                "requirement": r["requirement"],
                "compatible": compatible(r["requirement"], v["vehicle_type"]),
            })
    compatibility = pd.DataFrame(compatibility_rows)

    alpha_rows = []
    for _, p in patients.iterrows():
        for _, v in vehicles.iterrows():
            for phi in ["O", "B"]:
                alpha_rows.append({
                    "patient_id": p["patient_id"],
                    "vehicle_id": v["vehicle_id"],
                    "vehicle_type": v["vehicle_type"],
                    "service_phase_phi": phi,
                    "requirement": p["requirement"],
                    "alpha_pvk_phi": compatible(p["requirement"], v["vehicle_type"]),
                })
    compatibility_alpha = pd.DataFrame(alpha_rows)

    # Matrices
    distance_matrix_points = make_distance_matrix_points_vectorized(points)
    distance_matrix_points.to_csv(
        os.path.join(matrix_dir, "distance_matrix_points.csv.gz"),
        index=False,
        compression="gzip",
    )

    distance_matrix_km = distance_matrix_points.pivot(
        index="origin_point_id",
        columns="destination_point_id",
        values="distance_km",
    ).reset_index()
    distance_matrix_km.columns.name = None
    with pd.ExcelWriter(output_distance_matrix_excel, engine="openpyxl") as writer:
        distance_matrix_km.to_excel(writer, sheet_name="distance_matrix_km", index=False)

    arc_parameters = distance_matrix_points[
        distance_matrix_points["origin_point_id"] != distance_matrix_points["destination_point_id"]
    ].copy()
    arc_parameters["t_ij"] = (arc_parameters["distance_km"] / DEFAULT_SPEED_KMH * 60.0).round(2)
    arc_parameters["c_ij"] = (arc_parameters["distance_km"] * DEFAULT_COST_PER_KM).round(2)
    arc_parameters = arc_parameters[[
        "origin_point_id",
        "destination_point_id",
        "origin_point_type",
        "destination_point_type",
        "distance_km",
        "distance_m",
        "t_ij",
        "c_ij",
        "distance_source",
    ]]
    arc_parameters.to_csv(
        os.path.join(matrix_dir, "arc_parameters.csv.gz"),
        index=False,
        compression="gzip",
    )

    type_frames = []
    base_travel = distance_matrix_points[["origin_point_id", "destination_point_id", "distance_km"]].copy()
    for vehicle_type, speed in speed_by_type_kmh.items():
        tmp = base_travel.copy()
        tmp["vehicle_type"] = vehicle_type
        tmp["t_ij_type"] = (tmp["distance_km"] / float(speed) * 60.0).round(2)
        tmp["c_ij_type"] = (tmp["distance_km"] * float(cost_by_type_km[vehicle_type])).round(2)
        type_frames.append(tmp)
    travel_matrix_by_vehicle_type = pd.concat(type_frames, ignore_index=True)
    travel_matrix_by_vehicle_type.to_csv(
        os.path.join(matrix_dir, "travel_matrix_by_vehicle_type.csv.gz"),
        index=False,
        compression="gzip",
    )

    matrix_export_notes = pd.DataFrame([
        {
            "table": "travel_matrix_by_vehicle",
            "status": "exported" if EXPORT_VEHICLE_ID_MATRIX else "not_exported_by_default",
            "reason": "Set EXPORT_VEHICLE_ID_MATRIX=True if a full vehicle-id matrix is required. The type matrix is exported and vehicle ids inherit type speed/cost.",
        }
    ])

    if EXPORT_VEHICLE_ID_MATRIX:
        export_travel_matrix_by_vehicle_stream(
            distance_matrix_points=distance_matrix_points,
            vehicles=vehicles,
            output_path=os.path.join(matrix_dir, "travel_matrix_by_vehicle.csv.gz"),
        )

    request_direct_distance = requests[[
        "request_id",
        "patient_id",
        "trip_type",
        "origin_point_id",
        "destination_point_id",
        "hospital_id",
        "activity_type",
    ]].merge(
        distance_matrix_points,
        on=["origin_point_id", "destination_point_id"],
        how="left",
    )
    request_direct_distance.to_csv(
        os.path.join(matrix_dir, "request_direct_distance.csv"),
        index=False,
    )

    vehicle_type_summary = (
        vehicles
        .groupby("vehicle_type", as_index=False)
        .agg(
            eta_k=("vehicle_id", "count"),
            Q_k=("capacity_Qk", "first"),
            speed_kmh=("speed_kmh", "first"),
            cost_per_km=("cost_per_km", "first"),
        )
    )

    patient_parameters = patients[[
        "patient_id",
        "hospital_id",
        "activity_type",
        "appointment_minute",
        "appointment_time_hhmm",
        "hospital_stay_min",
        "return_ready_minute",
        "return_ready_time_hhmm",
        "max_ride_time_min",
        "waiting_time_after_treatment_min",
        "distance_to_assigned_hospital_km",
        "closest_hospital_id",
        "closest_hospital_distance_km",
        "assigned_hospital_is_coherent",
    ]].copy()
    patient_parameters = patient_parameters.rename(columns={
        "appointment_minute": "a_p",
        "hospital_stay_min": "tau_p",
        "max_ride_time_min": "Delta_p",
        "waiting_time_after_treatment_min": "Psi_p",
    })

    global_parameters = pd.DataFrame([
        {"symbol": "instance_category", "value": cfg["category"], "unit": "id", "description": "Generated instance category."},
        {"symbol": "target_patients", "value": n_patients_target, "unit": "count", "description": "Requested number of patients."},
        {"symbol": "target_vehicles", "value": n_vehicles_target, "unit": "count", "description": "Requested number of vehicles."},
        {"symbol": "target_hospitals", "value": n_hospitals_target, "unit": "count", "description": "Requested number of hospitals."},
        {"symbol": "s", "value": SERVICE_PICKUP_MIN, "unit": "minutes", "description": "Service time at each visited node."},
        {"symbol": "s_pickup", "value": SERVICE_PICKUP_MIN, "unit": "minutes", "description": "Pickup service time."},
        {"symbol": "s_dropoff", "value": SERVICE_DROPOFF_MIN, "unit": "minutes", "description": "Dropoff service time."},
        {"symbol": "Omega", "value": OMEGA_MAX_EARLY_ARRIVAL_MIN, "unit": "minutes", "description": "Maximum early arrival before appointment."},
        {"symbol": "Psi", "value": PSI_MAX_RETURN_DELAY_MIN, "unit": "minutes", "description": "Maximum delay after treatment completion for return pickup."},
        {"symbol": "Delta_global", "value": int(patients["max_ride_time_min"].max()), "unit": "minutes", "description": "Maximum observed ride-time limit."},
        {"symbol": "lambda", "value": LAMBDA_RIDE_TIME_WEIGHT, "unit": "objective_weight", "description": "Ride-time weight."},
        {"symbol": "M", "value": BIG_M, "unit": "model_constant", "description": "Big-M constant."},
        {"symbol": "default_speed_kmh", "value": DEFAULT_SPEED_KMH, "unit": "km/h", "description": "Default speed for t_ij."},
        {"symbol": "default_cost_per_km", "value": DEFAULT_COST_PER_KM, "unit": "currency/km", "description": "Default cost for c_ij."},
        {"symbol": "external_transport_share", "value": EXTERNAL_TRANSPORT_SHARE, "unit": "ratio", "description": "Share of external visits using the studied fleet."},
    ])

    hospital_daily_capacity = (
        hospital_activity_schedule[["hospital_id", "activity_type", "daily_capacity"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    hospital_capacity_summary = (
        hospital_activity_schedule
        .groupby(["hospital_id", "activity_type"], as_index=False)
        .agg(
            daily_capacity=("daily_capacity", "first"),
            total_slot_capacity=("slot_capacity", "sum"),
            max_slot_capacity=("slot_capacity", "max"),
            number_of_slots=("slot_time", "count"),
        )
    )
    hospital_capacity_summary["capacity_feasible"] = (
        hospital_capacity_summary["daily_capacity"] == hospital_capacity_summary["total_slot_capacity"]
    ).astype(int)
    hospital_capacity_summary = hospital_capacity_summary.merge(
        hospitals[["hospital_id", "hospital_name", "city", "hospital_type"]],
        on="hospital_id",
        how="left",
    )

    hospital_occupancy_table = hospital_activity_schedule.copy()
    hospital_occupancy_table["slot_start_datetime"] = hospital_occupancy_table["slot_start_min"].apply(
        lambda m: minute_to_datetime(SERVICE_DATE, int(m))
    )

    math_parameter_status = pd.DataFrame([
        {"symbol": "c_ij", "status": "included", "table": "arc_parameters", "column": "c_ij"},
        {"symbol": "t_ij", "status": "included", "table": "arc_parameters", "column": "t_ij"},
        {"symbol": "t_ij_type", "status": "included", "table": "travel_matrix_by_vehicle_type", "column": "t_ij_type"},
        {"symbol": "c_ij_type", "status": "included", "table": "travel_matrix_by_vehicle_type", "column": "c_ij_type"},
        {"symbol": "a_p", "status": "included", "table": "patient_parameters", "column": "a_p"},
        {"symbol": "tau_p", "status": "included", "table": "patient_parameters", "column": "tau_p"},
        {"symbol": "s", "status": "included", "table": "global_parameters / model_nodes", "column": "s / service_time_min"},
        {"symbol": "Q_k", "status": "included", "table": "vehicle_type_summary", "column": "Q_k"},
        {"symbol": "eta_k", "status": "included", "table": "vehicle_type_summary", "column": "eta_k"},
        {"symbol": "Omega", "status": "included", "table": "global_parameters", "column": "Omega"},
        {"symbol": "Psi", "status": "included", "table": "global_parameters / patient_parameters", "column": "Psi / Psi_p"},
        {"symbol": "Delta", "status": "included", "table": "patient_parameters", "column": "Delta_p"},
        {"symbol": "lambda", "status": "included", "table": "global_parameters", "column": "lambda"},
        {"symbol": "M", "status": "included", "table": "global_parameters", "column": "M"},
        {"symbol": "alpha_pvk_phi", "status": "included", "table": "compatibility_alpha", "column": "alpha_pvk_phi"},
    ])

    model_dictionary = pd.DataFrame([
        {"math_symbol": "P", "dataset_table": "patients", "dataset_column": "patient_id", "meaning": "Set of patients."},
        {"math_symbol": "H", "dataset_table": "hospitals", "dataset_column": "hospital_id", "meaning": "Set of hospitals."},
        {"math_symbol": "K", "dataset_table": "vehicles", "dataset_column": "vehicle_type", "meaning": "Set of vehicle types."},
        {"math_symbol": "V_k", "dataset_table": "vehicles", "dataset_column": "vehicle_id", "meaning": "Available vehicles."},
        {"math_symbol": "0", "dataset_table": "points", "dataset_column": "D0", "meaning": "Depot node."},
        {"math_symbol": "c_ij", "dataset_table": "arc_parameters", "dataset_column": "c_ij", "meaning": "Travel cost."},
        {"math_symbol": "t_ij", "dataset_table": "arc_parameters", "dataset_column": "t_ij", "meaning": "Travel time."},
        {"math_symbol": "a_p", "dataset_table": "patient_parameters", "dataset_column": "a_p", "meaning": "Appointment time."},
        {"math_symbol": "tau_p", "dataset_table": "patient_parameters", "dataset_column": "tau_p", "meaning": "Treatment duration."},
        {"math_symbol": "s", "dataset_table": "global_parameters / model_nodes", "dataset_column": "s / service_time_min", "meaning": "Service time."},
        {"math_symbol": "Q_k", "dataset_table": "vehicle_type_summary", "dataset_column": "Q_k", "meaning": "Vehicle capacity."},
        {"math_symbol": "eta_k", "dataset_table": "vehicle_type_summary", "dataset_column": "eta_k", "meaning": "Number of vehicles by type."},
        {"math_symbol": "Omega", "dataset_table": "global_parameters", "dataset_column": "Omega", "meaning": "Maximum early arrival."},
        {"math_symbol": "Psi", "dataset_table": "global_parameters / patient_parameters", "dataset_column": "Psi / Psi_p", "meaning": "Maximum return pickup delay."},
        {"math_symbol": "Delta", "dataset_table": "patient_parameters", "dataset_column": "Delta_p", "meaning": "Maximum ride time."},
        {"math_symbol": "lambda", "dataset_table": "global_parameters", "dataset_column": "lambda", "meaning": "Ride-time weight."},
        {"math_symbol": "M", "dataset_table": "global_parameters", "dataset_column": "M", "meaning": "Big-M constant."},
        {"math_symbol": "alpha_pvk_phi", "dataset_table": "compatibility_alpha", "dataset_column": "alpha_pvk_phi", "meaning": "Patient-vehicle-phase compatibility."},
    ])

    metadata = pd.DataFrame([
        {"field": "instance_category", "value": cfg["category"]},
        {"field": "instance_size", "value": cfg["size"]},
        {"field": "service_date", "value": SERVICE_DATE},
        {"field": "target_patients", "value": n_patients_target},
        {"field": "target_vehicles", "value": n_vehicles_target},
        {"field": "target_hospitals", "value": n_hospitals_target},
        {"field": "transport_horizon", "value": f"{min_to_hhmm(TRANSPORT_START_MIN)}-{min_to_hhmm(TRANSPORT_END_MIN)}"},
        {"field": "appointment_horizon", "value": f"{min_to_hhmm(APPOINTMENT_START_MIN)}-{min_to_hhmm(APPOINTMENT_END_MIN)}"},
        {"field": "n_hospitals", "value": len(hospitals)},
        {"field": "n_medical_appointments", "value": len(appointment_plan)},
        {"field": "n_patients", "value": len(patients)},
        {"field": "n_requests_total", "value": len(requests)},
        {"field": "n_outbound_requests", "value": int((requests["trip_type"] == "outbound").sum())},
        {"field": "n_return_requests_same_day", "value": int((requests["trip_type"] == "return").sum())},
        {"field": "n_future_returns", "value": len(future_returns)},
        {"field": "n_vehicles", "value": len(vehicles)},
        {"field": "external_transport_share", "value": EXTERNAL_TRANSPORT_SHARE},
        {"field": "patient_distribution_method", "value": "hospital_share_then_specialty_share_then_slot_share"},
        {"field": "distance_method", "value": "haversine_x_road_factor"},
        {"field": "seed", "value": seed},
        {"field": "important_note", "value": "Renaison and third dialysis centre volumes are assumptions to validate."},
    ])

    # Checks
    if len(patients) != n_patients_target:
        raise RuntimeError(f"{instance_name}: patient count mismatch.")
    if len(vehicles) != n_vehicles_target:
        raise RuntimeError(f"{instance_name}: vehicle count mismatch.")
    if len(hospitals) != n_hospitals_target:
        raise RuntimeError(f"{instance_name}: hospital count mismatch.")
    if int(hospital_distribution["target_patients"].sum()) != n_patients_target:
        raise RuntimeError(f"{instance_name}: hospital distribution total mismatch.")
    if int(hospital_specialty_distribution["target_patients"].sum()) != n_patients_target:
        raise RuntimeError(f"{instance_name}: specialty distribution total mismatch.")
    invalid_request_times = requests[
        (requests["earliest_pickup_min"] < TRANSPORT_START_MIN)
        | (requests["latest_dropoff_min"] > TRANSPORT_END_MIN)
    ]
    if len(invalid_request_times) > 0:
        raise RuntimeError(f"{instance_name}: some requests are outside the transport horizon.")

    # Export CSV + Excel
    instance_config = pd.DataFrame([cfg])
    tables_excel = {
        "instance_config": instance_config,
        "metadata": metadata,
        "global_parameters": global_parameters,
        "math_parameter_status": math_parameter_status,
        "matrix_export_notes": matrix_export_notes,
        "hospitals": hospitals,
        "hospital_distribution": hospital_distribution,
        "hospital_specialty_dist": hospital_specialty_distribution,
        "hospital_activity_schedule": hospital_activity_schedule,
        "hospital_daily_capacity": hospital_daily_capacity,
        "hospital_capacity_summary": hospital_capacity_summary,
        "hospital_occupancy": hospital_occupancy_table,
        "appointment_plan": appointment_plan,
        "patients": patients,
        "patient_parameters": patient_parameters,
        "requests": requests,
        "future_returns": future_returns,
        "vehicles": vehicles,
        "vehicle_type_summary": vehicle_type_summary,
        "points": points,
        "model_nodes": model_nodes,
        "compatibility": compatibility,
        "compatibility_alpha": compatibility_alpha,
        "request_direct_distance": request_direct_distance,
        "model_dictionary": model_dictionary,
    }

    for name, df in tables_excel.items():
        df.to_csv(os.path.join(csv_dir, f"{name}.csv"), index=False)

    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
        for name, df in tables_excel.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)

    summary = {
        "instance": instance_name,
        "category": cfg["category"],
        "size": cfg["size"],
        "patients": len(patients),
        "vehicles": len(vehicles),
        "hospitals": len(hospitals),
        "requests_total": len(requests),
        "future_returns": len(future_returns),
        "points": len(points),
        "folder": base_dir,
        "excel": output_excel,
    }
    for _, distribution_row in hospital_distribution.iterrows():
        hospital_id = str(distribution_row["hospital_id"])
        summary[f"{hospital_id}_patients"] = int(distribution_row["target_patients"])
        summary[f"{hospital_id}_share_pct"] = float(distribution_row["actual_share_pct"])

    # Release large objects before next instance
    del distance_matrix_points, distance_matrix_km, arc_parameters, travel_matrix_by_vehicle_type
    gc.collect()

    return summary

# ------------------------------------------------------------
# Batch run: all instances + final ZIP
# ------------------------------------------------------------
if os.path.exists(ROOT_DIR):
    shutil.rmtree(ROOT_DIR)
os.makedirs(ROOT_DIR, exist_ok=True)

manifest_rows = []
for idx, cfg in enumerate(INSTANCE_GRID.to_dict("records"), start=1):
    print(f"[{idx}/{len(INSTANCE_GRID)}] Generating {cfg['category']} ...")
    manifest_rows.append(generate_one_instance(cfg, idx))

manifest = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(ROOT_DIR, "manifest_instances.csv")
manifest.to_csv(manifest_path, index=False)

zip_base = ALL_OUTPUT_ZIP[:-4] if ALL_OUTPUT_ZIP.endswith(".zip") else ALL_OUTPUT_ZIP
if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")
shutil.make_archive(zip_base, "zip", ROOT_DIR)

print("\nALL INSTANCES GENERATED SUCCESSFULLY")
print("Root folder:", ROOT_DIR)
print("Manifest:", manifest_path)
print("ZIP:", zip_base + ".zip")
display(manifest)

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception:
    print("Automatic download not available. You can find the ZIP here:", zip_base + ".zip")

[1/48] Generating E/20/5/1 ...
[2/48] Generating E/20/5/2 ...
[3/48] Generating E/20/10/1 ...
[4/48] Generating E/20/10/2 ...
[5/48] Generating E/50/5/1 ...
[6/48] Generating E/50/5/2 ...
[7/48] Generating E/50/10/1 ...
[8/48] Generating E/50/10/2 ...
[9/48] Generating E/75/5/1 ...
[10/48] Generating E/75/5/2 ...
[11/48] Generating E/75/10/1 ...
[12/48] Generating E/75/10/2 ...
[13/48] Generating S/100/15/2 ...
[14/48] Generating S/100/15/3 ...
[15/48] Generating S/100/20/2 ...
[16/48] Generating S/100/20/3 ...
[17/48] Generating S/200/15/2 ...
[18/48] Generating S/200/15/3 ...
[19/48] Generating S/200/20/2 ...
[20/48] Generating S/200/20/3 ...
[21/48] Generating S/250/15/2 ...
[22/48] Generating S/250/15/3 ...
[23/48] Generating S/250/20/2 ...
[24/48] Generating S/250/20/3 ...
[25/48] Generating M/300/20/3 ...
[26/48] Generating M/300/30/3 ...
[27/48] Generating M/300/40/3 ...
[28/48] Generating M/350/20/3 ...
[29/48] Generating M/350/30/3 ...
[30/48] Generating M/350/40/3 ...
[31/48]

,instance,category,size,patients,vehicles,hospitals,requests_total,future_returns,points,folder,excel,H1_patients,H1_share_pct,H2_patients,H2_share_pct,H3_patients,H3_share_pct
0,E_20_5_1,E/20/5/1,E,20,5,1,35,5,27,/content/darp_roanne_instances/E_20_5_1,/content/darp_roanne_instances/E_20_5_1/E_20_5...,20,100.0000,NaN,NaN,NaN,NaN
1,E_20_5_2,E/20/5/2,E,20,5,2,36,4,28,/content/darp_roanne_instances/E_20_5_2,/content/darp_roanne_instances/E_20_5_2/E_20_5...,13,65.0000,7.0,35.0000,NaN,NaN
2,E_20_10_1,E/20/10/1,E,20,10,1,34,6,32,/content/darp_roanne_instances/E_20_10_1,/content/darp_roanne_instances/E_20_10_1/E_20_...,20,100.0000,NaN,NaN,NaN,NaN
3,E_20_10_2,E/20/10/2,E,20,10,2,36,4,33,/content/darp_roanne_instances/E_20_10_2,/content/darp_roanne_instances/E_20_10_2/E_20_...,13,65.0000,7.0,35.0000,NaN,NaN
4,E_50_5_1,E/50/5/1,E,50,5,1,87,13,57,/content/darp_roanne_instances/E_50_5_1,/content/darp_roanne_instances/E_50_5_1/E_50_5...,50,100.0000,NaN,NaN,NaN,NaN
5,E_50_5_2,E/50/5/2,E,50,5,2,88,12,58,/content/darp_roanne_instances/E_50_5_2,/content/darp_roanne_instances/E_50_5_2/E_50_5...,32,64.0000,18.0,36.0000,NaN,NaN
6,E_50_10_1,E/50/10/1,E,50,10,1,87,13,62,/content/darp_roanne_instances/E_50_10_1,/content/darp_roanne_instances/E_50_10_1/E_50_...,50,100.0000,NaN,NaN,NaN,NaN
7,E_50_10_2,E/50/10/2,E,50,10,2,87,13,63,/content/darp_roanne_instances/E_50_10_2,/content/darp_roanne_instances/E_50_10_2/E_50_...,32,64.0000,18.0,36.0000,NaN,NaN
8,E_75_5_1,E/75/5/1,E,75,5,1,131,19,82,/content/darp_roanne_instances/E_75_5_1,/content/darp_roanne_instances/E_75_5_1/E_75_5...,75,100.0000,NaN,NaN,NaN,NaN
9,E_75_5_2,E/75/5/2,E,75,5,2,132,18,83,/content/darp_roanne_instances/E_75_5_2,/content/darp_roanne_instances/E_75_5_2/E_75_5...,48,64.0000,27.0,36.0000,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All model-input checks passed.
D0 -> H3 distance_km: 5.11674
Excel file: /content/darp_roanne_model_input/darp_roanne_model_input.xlsx
ZIP file: /content/darp_roanne_model_input.zip


,symbol,value,unit,description
0,s,5.0,minutes,Service time at each visited node.
1,Omega,45.0,minutes,Maximum allowable early arrival before appoint...
2,Psi,35.0,minutes,Maximum allowable delay after treatment comple...
3,lambda,1.0,objective_weight,Weight assigned to patient ride time in the ob...
4,M,100000.0,model_constant,Sufficiently large positive constant used in B...


,hospital_id,hospital_name,city,lat,lon
0,H1,Centre Hospitalier de Roanne,Roanne,46.053300,4.072700
1,H2,Clinique du Renaison,Roanne,46.045700,4.068800
2,H3,Centre Hospitalier de Roanne - Site de Bonvert,Mably,46.086667,4.065278


,patient_id,home_point_id,hospital_id,activity_type,requirement,a_p,a_p_hhmm,a_p_datetime,tau_p,Psi_p,Delta_p,sharing_possible
0,P1,HOME_P1,H1,dialysis,vsl_required,780,13:00,2026-05-18 13:00:00,240,35,25,0
1,P2,HOME_P2,H1,dialysis,prescription_medicale,480,08:00,2026-05-18 08:00:00,240,35,25,1
2,P3,HOME_P3,H1,dialysis,prescription_medicale,480,08:00,2026-05-18 08:00:00,240,35,25,1
3,P4,HOME_P4,H1,dialysis,vsl_required,480,08:00,2026-05-18 08:00:00,240,35,25,1
4,P5,HOME_P5,H1,dialysis,vsl_required,480,08:00,2026-05-18 08:00:00,240,35,25,1


,appointment_id,patient_id,hospital_id,activity_type,requirement,appointment_minute_a_p,appointment_time_hhmm,appointment_datetime,treatment_duration_tau_p,sharing_possible
0,A0001,P1,H1,dialysis,vsl_required,780,13:00,2026-05-18 13:00:00,240,0
1,A0002,P2,H1,dialysis,prescription_medicale,480,08:00,2026-05-18 08:00:00,240,1
2,A0003,P3,H1,dialysis,prescription_medicale,480,08:00,2026-05-18 08:00:00,240,1
3,A0004,P4,H1,dialysis,vsl_required,480,08:00,2026-05-18 08:00:00,240,1
4,A0005,P5,H1,dialysis,vsl_required,480,08:00,2026-05-18 08:00:00,240,1


...

,vehicle_type_k,Q_k,eta_k
0,taxi_conventionne,3,10
1,vsl,3,7
2,vtc_premium,3,3


,origin_point_id,D0,H1,H2,H3,HOME_P1,HOME_P10,HOME_P100,HOME_P101,HOME_P102,...,HOME_P90,HOME_P91,HOME_P92,HOME_P93,HOME_P94,HOME_P95,HOME_P96,HOME_P97,HOME_P98,HOME_P99
0,D0,0.00000,0.52747,0.60878,5.11674,8.50614,4.45354,9.97347,12.70444,12.42050,...,11.32057,10.55266,14.26159,5.99580,14.82694,13.70440,7.92838,8.34886,12.53512,14.34851
1,H1,0.52747,0.00000,1.12135,4.69266,8.01036,4.64822,9.78056,13.19939,12.93794,...,11.84406,11.01786,14.09842,5.96719,14.90008,13.58621,7.72626,7.97514,12.45528,14.29231
2,H2,0.60878,1.12135,0.00000,5.70423,9.11487,4.49587,10.03691,12.23911,11.81677,...,10.72453,9.95401,14.64143,6.27050,14.96190,13.66904,8.35491,8.64941,12.83700,14.62365
3,H3,5.11674,4.69266,5.70423,0.00000,3.81995,5.77366,12.10624,16.17260,17.23982,...,16.03444,15.65112,10.57919,4.73577,13.24449,16.23235,4.41558,8.86207,9.65668,11.57941
4,HOME_P1,8.50614,8.01036,9.11487,3.81995,0.00000,9.54574,12.70405,19.98452,20.87833,...,19.71771,18.99486,11.18598,8.07695,15.43041,16.78640,6.48961,8.79992,11.19369,13.01080


,request_id,patient_id,service_phase_phi,trip_type,requirement,alpha_taxi_conventionne,alpha_vsl,alpha_vtc_premium
0,R0001,P1,O,outbound,vsl_required,0,1,0
1,R0002,P1,B,return,vsl_required,0,1,0
2,R0003,P2,O,outbound,prescription_medicale,1,1,0
3,R0004,P2,B,return,prescription_medicale,1,1,0
4,R0005,P3,O,outbound,prescription_medicale,1,1,0


,requirement,taxi_conventionne,vsl,vtc_premium
0,standard_assis,1,1,1
1,prescription_medicale,1,1,0
2,vsl_required,0,1,0
3,premium,1,0,1


,activity_type,share_non_sharing,n_eligible_patients,n_non_sharing_patients
0,dialysis,0.05,83,5
1,therapeutic,0.10,8,1
2,rehabilitation,0.10,24,3


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/image(82).png'